<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 자연어 처리와 LLM · 06. T5

## Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer

- **원 논문:** [Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://arxiv.org/abs/1910.10683)
- **저자 / 발표:** Colin Raffel et al. · JMLR 2020 (2019)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** text-to-text 형식, span sentinel target, relative position bias를 재현한다.

**축소하거나 생략한 부분:** C4와 최대 11B parameter 대신 60개 3-token sequence와 작은 Transformer를 쓰며, 48개 train/12개 held-out 행을 분리한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1.4 and Fig. 2: span replacement with sentinel tokens | Task 1 | sentinel input/target sequence |
| §2.1: encoder-decoder attention and relative position embeddings | Task 2 explicit encoder/causal decoder/cross-attention | bias shape, nonzero gradient, and future invariance |
| §3.3.4: span-corruption objective | Task 3 | denoising accuracy |

## 핵심 아이디어와 수식

$$L_{span}=-\sum_t\log p(y_t\mid y_{<t},\widetilde{x})$$

**기호 해설:** x-tilde는 sentinel로 span을 바꾼 입력, y는 sentinel과 제거 span을 잇는 target이다.

**코드 대응:** Task 1이 span pair, Task 2가 logits에 relative bias를 더하는 encoder/causal decoder/cross-attention, Task 3이 denoising CE를 맡는다.

**입력과 출력 shape:** train source [48,2], decoder [48,4]와 held-out source [12,2]를 분리해 logits [B,4,V]와 token accuracy를 계산한다.

**포트폴리오 구현 범위:** 여러 span과 bucketed bias 대신 한 span/clip distance를 쓰되 text-to-text 목적은 유지한다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1.4 and Fig. 2: span replacement with sentinel tokens
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** sentinel input/target sequence

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 1: 설정과 source [a,b,c]의 sentinel span corruption을 구현하세요.
@dataclass
class T5Config:
    pass


def make_span_corruption(source, sentinel_id, eos_id, bos_id):
    raise NotImplementedError

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.1: encoder-decoder attention and relative position embeddings
- **노트북에서 구현할 부분:** Task 2 explicit encoder/causal decoder/cross-attention
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** bias shape, nonzero gradient, and future invariance

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 2: relative bias class와 encoder-decoder T5Mini를 구현하세요.
class RelativePositionBias(nn.Module):
    """Clipped relative distance를 head별 attention bias로 바꾼다."""

    def __init__(self, heads, max_distance):
        super().__init__()

    def forward(self, query_length, key_length, device):
        """Head별 bias [H, T_q, T_k]를 반환한다."""
        raise NotImplementedError


class ExplicitMultiHeadAttention(nn.Module):
    """Relative bias와 mask를 score에 적용하는 attention이다."""

    def __init__(self, model_dim, heads):
        super().__init__()

    def split_heads(self, tensor):
        """[B, T, D]를 [B, H, T, d_h]로 분할한다."""
        raise NotImplementedError

    def merge_heads(self, tensor):
        """[B, H, T, d_h]를 [B, T, D]로 병합한다."""
        raise NotImplementedError

    def forward(
        self,
        query,
        key,
        value,
        relative_bias=None,
        attention_mask=None,
    ):
        """Bias와 mask가 적용된 attention output과 weights를 반환한다."""
        raise NotImplementedError


class T5EncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

    def forward(self, hidden, relative_bias):
        """Encoder hidden과 self-attention weights를 반환한다."""
        raise NotImplementedError


class T5DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

    def forward(self, hidden, memory, relative_bias, causal_mask):
        """Decoder hidden과 self/cross attention weights를 반환한다."""
        raise NotImplementedError


class T5Mini(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()

    def causal_mask(self, length, device):
        """미래 token을 가리는 bool mask [T, T]를 만든다."""
        raise NotImplementedError

    def forward(self, source, decoder_inputs):
        raise NotImplementedError

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.3.4: span-corruption objective
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** denoising accuracy

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 3: denoising loss, train step, token accuracy를 구현하세요.
def denoising_loss(logits, targets):
    raise NotImplementedError


def train_t5_step(model, optimizer, source, decoder_inputs, targets):
    raise NotImplementedError


def evaluate_t5(model, source, decoder_inputs, targets):
    raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.